# 第 4 章 形状变换与数组拼接 (Reshape & Concatenate)

> 🎯 **本章目标**
>
> - 建立 **axis（轴）** 心智模型：axis 就是维度编号，这是全章（乃至第 5 章）的地基
> - 掌握 `reshape` 的规则：改变形状不改变数据，`-1` 自动推断
> - 分清 `ravel` / `flatten` / `reshape(-1)`：谁返回视图、谁返回拷贝
> - 学会转置家族 `.T` / `transpose` / `swapaxes` / `moveaxis`
> - 学会用 `newaxis` / `expand_dims` / `squeeze` 增删维度，参与广播
> - 分清 `concatenate`（沿已有轴）与 `stack`（沿新轴）的本质区别
> - 学会拆分与 `tile` / `repeat`，并完成一个「批量数据整形」实战

> 📖 前置知识：第 1 章 ndarray 基础、第 2 章索引切片、第 3 章广播机制。

有了第 3 章的运算能力，本章我们把「形状」这一维玩到极致——改形状、转置、拼接、拆分，随心所欲。

In [1]:
import numpy as np
print(np.__version__)

2.4.4


## 4.1 axis（轴）心智模型总图

### 为什么先讲 axis？

后面的 `concatenate(axis=?)`、`max(axis=?)`、`squeeze(axis=?)` 全都问同一个问题：
**沿着哪个方向操作？**

一句话直觉：**axis 就是维度的编号**。

- `axis=0` → 第 0 维（最外层）
- `axis=1` → 第 1 维
- `axis=2` → 第 2 维（最内层）

对三维数组 `(2, 3, 4)`：

- `shape[0] = 2`：axis=0 方向有 2 个「块」
- `shape[1] = 3`：axis=1 方向有 3 个「行」
- `shape[2] = 4`：axis=2 方向有 4 个「列」

```mermaid
flowchart TD
    A["np.zeros((2, 3, 4))"] --> B["axis=0: 第1维, 长度 2 → 2 个深度切片"]
    A --> C["axis=1: 第2维, 长度 3 → 每片 3 行"]
    A --> D["axis=2: 第3维, 长度 4 → 每行 4 列"]
    B --> M["axis = 维度编号<br/>沿 axis=k 操作 = 沿第 k 维方向"]
    C --> M
    D --> M
```

> 💡 **记法**：`arr[i, j, k]` 里三个下标的位置，从左到右正好对应 axis=0、1、2。

In [2]:
# 用 3D 数组建立 axis 心智模型
arr3d = np.arange(24).reshape(2, 3, 4)   # 2 块 × 3 行 × 4 列
print("arr3d.shape =", arr3d.shape)
print("arr3d:\n", arr3d)
print("arr3d[0] 是 axis=0 方向的第 1 块, 形状:", arr3d[0].shape)
print("arr3d[0, 1] 是 axis=1 方向的第 2 行, 形状:", arr3d[0, 1].shape)
print("arr3d[0, 1, 2] 是 axis=2 方向第 3 列的元素:", arr3d[0, 1, 2])

arr3d.shape = (2, 3, 4)
arr3d:
 [[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]
arr3d[0] 是 axis=0 方向的第 1 块, 形状: (3, 4)
arr3d[0, 1] 是 axis=1 方向的第 2 行, 形状: (4,)
arr3d[0, 1, 2] 是 axis=2 方向第 3 列的元素: 6


## 4.2 reshape：改变形状，不改变数据

### 解决什么问题？

数据还是那些数据，只是把它们「重新排列」成另一种形状。
例如 6 个数 `0,1,2,3,4,5` 可以排成 2 行 3 列，也可以排成 3 行 2 列。

### 关键要点

1. **不改变数据**：元素个数必须不变（6 个元素不能 reshape 成 `(2, 4)`，因为 2×4≠6）
2. **`-1` 自动推断**：`reshape(-1)` 展平；`reshape(3, -1)` 由 numpy 算出另一个维度
3. **通常返回视图**：reshape 一般**不复制数据**，而是给同一块内存换一个「读法」

### 行优先（C order）vs 列优先（F order）

「按什么顺序填充」由 order 决定：

| order | 中文 | 填充顺序 | `np.arange(6).reshape(2, 3)` |
| --- | --- | --- | --- |
| `'C'` | 行优先 | 先填满一行再下一行 | `[[0,1,2],[3,4,5]]` |
| `'F'` | 列优先 | 先填满一列再下一列 | `[[0,2,4],[1,3,5]]` |

> 💡 C 语言、Python 默认都是**行优先**（最右边的维度变化最快）。了解 F order 即可，日常几乎都用 C order。

In [3]:
# reshape: 改变形状, 不改变数据
arr = np.arange(6)
m = arr.reshape(2, 3)
print("arr =", arr, "| shape:", arr.shape)
print("arr.reshape(2,3) =\n", m)
print("m.shape =", m.shape)
print("数据顺序没变: 仍按行读是 0,1,2,3,4,5")

arr = [0 1 2 3 4 5] | shape: (6,)
arr.reshape(2,3) =
 [[0 1 2]
 [3 4 5]]
m.shape = (2, 3)
数据顺序没变: 仍按行读是 0,1,2,3,4,5


In [4]:
# 用 -1 让 numpy 自动推断维度
arr = np.arange(12)
print("arr.reshape(3, -1).shape =", arr.reshape(3, -1).shape)   # -1 -> 4
print("arr.reshape(-1, 6).shape =", arr.reshape(-1, 6).shape)   # -1 -> 2
print("arr.reshape(-1).shape     =", arr.reshape(-1).shape)     # 展平成一维

arr.reshape(3, -1).shape = (3, 4)
arr.reshape(-1, 6).shape = (2, 6)
arr.reshape(-1).shape     = (12,)


In [5]:
# reshape 返回视图: 修改结果会影响原数组
base = np.arange(6)
view = base.reshape(2, 3)
print("np.shares_memory(base, view) =", np.shares_memory(base, view))  # True
view[0, 0] = 999
print("修改 view 后 base =", base)   # 原数组也被改了 (共享同一块内存)

np.shares_memory(base, view) = True
修改 view 后 base = [999   1   2   3   4   5]


In [6]:
# C order(行优先) vs F order(列优先)
arr = np.arange(6)
c_ord = arr.reshape(2, 3, order='C')   # 先填满一行
f_ord = arr.reshape(2, 3, order='F')   # 先填满一列
print("order='C' (行优先):\n", c_ord)
print("order='F' (列优先):\n", f_ord)

order='C' (行优先):
 [[0 1 2]
 [3 4 5]]
order='F' (列优先):
 [[0 2 4]
 [1 3 5]]


## 4.3 ravel / flatten / reshape(-1)：三种展平方式

### 解决什么问题？

把多维数组变成一维。三个 API 长得像、行为有微妙区别，是面试/考试高频对比题。

| API | 返回视图还是拷贝 | 说明 |
| --- | --- | --- |
| `arr.ravel()` | **视图**（尽量） | 不复制数据，改它会改原数组 |
| `arr.flatten()` | **拷贝**（总是） | 永远新建数组，改了不影响原数组 |
| `arr.reshape(-1)` | 视图（尽量） | 与 ravel 类似，写法更「标准」 |

> 💡 **记忆**：`flatten` 带 `f`，像是把数组「压扁复印一份」——拷贝；`ravel` 不带 f，省事不复制——视图。
>
> ⚠️ **注意**：三种方法在**默认连续**的数组上都会返回视图；只有当原数组是非连续内存（如转置后的 `.T`）时，`ravel` 才会被迫复制。

In [7]:
# ravel / flatten / reshape(-1) 都能展平
arr = np.arange(6).reshape(2, 3)
print("arr =\n", arr)
print("arr.ravel()     =", arr.ravel(),     "| 与 arr 共享内存:", np.shares_memory(arr, arr.ravel()))
print("arr.flatten()   =", arr.flatten(),   "| 与 arr 共享内存:", np.shares_memory(arr, arr.flatten()))
print("arr.reshape(-1) =", arr.reshape(-1), "| 与 arr 共享内存:", np.shares_memory(arr, arr.reshape(-1)))

arr =
 [[0 1 2]
 [3 4 5]]
arr.ravel()     = [0 1 2 3 4 5] | 与 arr 共享内存: True
arr.flatten()   = [0 1 2 3 4 5] | 与 arr 共享内存: False
arr.reshape(-1) = [0 1 2 3 4 5] | 与 arr 共享内存: True


In [8]:
# 视图 vs 拷贝的实际影响
arr = np.arange(6).reshape(2, 3)
r = arr.ravel()      # 视图
f = arr.flatten()    # 拷贝

r[0] = 100           # 修改视图
print("修改 ravel 后 arr =", arr)    # arr 被影响了 (共享内存)
f[1] = 999           # 修改拷贝
print("修改 flatten 后 arr =", arr)  # arr 不受影响

修改 ravel 后 arr = [[100   1   2]
 [  3   4   5]]
修改 flatten 后 arr = [[100   1   2]
 [  3   4   5]]


## 4.4 转置家族：.T / transpose / swapaxes / moveaxis

### 解决什么问题？

把数据的「维度的顺序」重新排列。

| API | 作用 | 例子 |
| --- | --- | --- |
| `arr.T` | 翻转所有轴（转置） | `(2,3) → (3,2)`；`(2,3,4) → (4,3,2)` |
| `arr.transpose(2, 0, 1)` | 按给定顺序重排轴 | `(2,3,4) → (4,2,3)` |
| `np.swapaxes(a, 0, 2)` | 交换两个轴 | `(2,3,4) → (4,3,2)` |
| `np.moveaxis(a, 0, -1)` | 把一个轴移动到新位置 | `(2,3,4) → (3,4,2)` |

> 💡 这些操作**默认都是视图**（不复制数据），只是换一种方式「读」同一块内存。
>
> ⚠️ **陷阱**：转置会打乱内存的连续性，之后若再对转置结果做 `ravel()`，NumPy 会**被迫复制**一份连续数据。

In [9]:
# .T 转置: 翻转所有轴
a2d = np.arange(6).reshape(2, 3)
print("a2d =\n", a2d)
print("a2d.T =\n", a2d.T)
print("a2d.T.shape =", a2d.T.shape)
print(".T 是视图:", np.shares_memory(a2d, a2d.T))

a2d =
 [[0 1 2]
 [3 4 5]]
a2d.T =
 [[0 3]
 [1 4]
 [2 5]]
a2d.T.shape = (3, 2)
.T 是视图: True


In [10]:
# 高维转置: 用 axes 参数指定轴的新顺序
a3d = np.arange(24).reshape(2, 3, 4)
print("a3d.shape =", a3d.shape)
print("a3d.transpose(2, 0, 1).shape =", a3d.transpose(2, 0, 1).shape)  # (4,2,3)
print("a3d.transpose(1, 2, 0).shape =", a3d.transpose(1, 2, 0).shape)  # (3,4,2)

a3d.shape = (2, 3, 4)
a3d.transpose(2, 0, 1).shape = (4, 2, 3)
a3d.transpose(1, 2, 0).shape = (3, 4, 2)


In [11]:
# swapaxes 交换两个轴; moveaxis 移动一个轴
a3d = np.arange(24).reshape(2, 3, 4)
print("a3d.shape =", a3d.shape)
print("np.swapaxes(a3d, 0, 2).shape  =", np.swapaxes(a3d, 0, 2).shape)   # (4,3,2)
print("np.moveaxis(a3d, 0, -1).shape =", np.moveaxis(a3d, 0, -1).shape)  # (3,4,2)

a3d.shape = (2, 3, 4)
np.swapaxes(a3d, 0, 2).shape  = (4, 3, 2)
np.moveaxis(a3d, 0, -1).shape = (3, 4, 2)


## 4.5 增删维度：newaxis / expand_dims / squeeze

### 解决什么问题？

有些运算要求维度匹配（比如广播），但你的数据维度「不够」或「多了」。
这一节教你怎么插入 / 删除长度为 1 的轴。

| API | 作用 | 例子（`v` 形状 `(3,)`） |
| --- | --- | --- |
| `v[np.newaxis, :]` 或 `v[None, :]` | 在指定位置插入长度为 1 的轴 | `(3,) → (1, 3)` 行向量 |
| `v[:, np.newaxis]` | 同上，换一个位置 | `(3,) → (3, 1)` 列向量 |
| `np.expand_dims(v, axis)` | 函数式写法 | `expand_dims(v, 1) → (3, 1)` |
| `np.squeeze(arr)` | 删除所有长度为 1 的轴 | `(1,3,1,4) → (3,4)` |

> 💡 **典型用途**：把一维向量变成「列向量」，就能借助第 3 章的广播做外积等操作。
> 一个经典例子：`v[:, None] * w[None, :]` 用两个向量直接生成外积矩阵。

In [12]:
# np.newaxis / None: 在指定位置插入长度为 1 的新轴
v = np.array([1, 2, 3])            # 形状 (3,)
row = v[np.newaxis, :]             # (1, 3) 行向量
col = v[:, np.newaxis]             # (3, 1) 列向量
print("v.shape =", v.shape)
print("v[np.newaxis, :].shape =", row.shape, "| 行向量")
print("v[:, np.newaxis].shape =", col.shape, "| 列向量")
print("v[:, None] 与 v[:, np.newaxis] 等价:", np.array_equal(v[:, None], col))

v.shape = (3,)
v[np.newaxis, :].shape = (1, 3) | 行向量
v[:, np.newaxis].shape = (3, 1) | 列向量
v[:, None] 与 v[:, np.newaxis] 等价: True


In [13]:
# expand_dims / squeeze 与 newaxis 互补
v = np.array([1, 2, 3])
print("np.expand_dims(v, 0).shape =", np.expand_dims(v, 0).shape)  # (1,3)
print("np.expand_dims(v, 1).shape =", np.expand_dims(v, 1).shape)  # (3,1)

arr = np.ones((1, 3, 1, 4))
print("arr.shape =", arr.shape)
print("np.squeeze(arr).shape         =", np.squeeze(arr).shape)          # (3,4)
print("np.squeeze(arr, axis=2).shape =", np.squeeze(arr, axis=2).shape)  # (1,3,4)

np.expand_dims(v, 0).shape = (1, 3)
np.expand_dims(v, 1).shape = (3, 1)
arr.shape = (1, 3, 1, 4)
np.squeeze(arr).shape         = (3, 4)
np.squeeze(arr, axis=2).shape = (1, 3, 4)


In [14]:
# 典型用途: 用 newaxis 把向量变成行/列向量, 参与广播(衔接第3章)
scores = np.array([90, 80, 70])          # 3 门课成绩, (3,)
weights = np.array([0.2, 0.3, 0.5])      # 3 门课权重, (3,)
outer = scores[:, np.newaxis] * weights[np.newaxis, :]   # (3,1)*(1,3)->(3,3)
print("外积矩阵 (3,3):\n", outer)
print("outer.shape =", outer.shape)

外积矩阵 (3,3):
 [[18. 27. 45.]
 [16. 24. 40.]
 [14. 21. 35.]]
outer.shape = (3, 3)


## 4.6 拼接：concatenate / vstack / hstack / dstack / column_stack

### 解决什么问题？

把多个数组「接到一起」。关键问题同样是：**沿哪个轴接？**

```mermaid
flowchart LR
    A["数组 A (2,3)"] --> C["concatenate axis=0<br/>沿行方向堆叠 → (4,3)"]
    B["数组 B (2,3)"] --> C
    A --> D["concatenate axis=1<br/>沿列方向拼接 → (2,6)"]
    B --> D
```

| API | 方向 | 等价于 |
| --- | --- | --- |
| `np.concatenate([a, b], axis=0)` | 沿任意已有轴 | — |
| `np.vstack([a, b])` | 垂直堆叠（上下） | `concatenate(axis=0)` |
| `np.hstack([a, b])` | 水平拼接（左右） | `concatenate(axis=1)`（二维时） |
| `np.dstack([a, b])` | 沿深度（第 3 维） | `concatenate(axis=2)` |
| `np.column_stack([a, b])` | 按列拼接 | 二维时 ≈ hstack |

> ⚠️ **陷阱**：`concatenate` 要求**除拼接轴外，其他维度必须一致**。
> 例如两个 `(2,3)` 沿 axis=0 拼成 `(4,3)` 可以，但不能拿 `(2,3)` 和 `(2,4)` 沿 axis=0 拼。

In [15]:
# concatenate: 沿已有轴拼接
a = np.arange(6).reshape(2, 3)
b = np.arange(6, 12).reshape(2, 3)
print("a =\n", a)
print("b =\n", b)

c0 = np.concatenate([a, b], axis=0)   # 沿行: 上下堆 -> (4,3)
c1 = np.concatenate([a, b], axis=1)   # 沿列: 左右拼 -> (2,6)
print("concatenate axis=0 形状:", c0.shape)
print("concatenate axis=1 形状:", c1.shape)
print("axis=0 结果:\n", c0)
print("axis=1 结果:\n", c1)

a =
 [[0 1 2]
 [3 4 5]]
b =
 [[ 6  7  8]
 [ 9 10 11]]
concatenate axis=0 形状: (4, 3)
concatenate axis=1 形状: (2, 6)
axis=0 结果:
 [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
axis=1 结果:
 [[ 0  1  2  6  7  8]
 [ 3  4  5  9 10 11]]


In [16]:
# vstack / hstack / dstack / column_stack 快捷方式
a = np.arange(6).reshape(2, 3)
b = np.arange(6, 12).reshape(2, 3)
print("vstack 形状:", np.vstack([a, b]).shape)                  # (4,3)
print("hstack 形状:", np.hstack([a, b]).shape)                  # (2,6)
print("dstack 形状:", np.dstack([a, b]).shape)                  # (2,3,2)
print("column_stack 形状:", np.column_stack([a, b]).shape)      # (2,6)

vstack 形状: (4, 3)
hstack 形状: (2, 6)
dstack 形状: (2, 3, 2)
column_stack 形状: (2, 6)


## 4.7 stack 家族：沿「新轴」堆叠

### 核心区别：stack vs concatenate

`concatenate` 沿**已有轴**拼接，维度不变；
`stack` 会**新建一个轴**，维度 +1。

| | `np.concatenate` | `np.stack` |
| --- | --- | --- |
| 沿什么拼接 | 已有的轴 | **新建的轴** |
| 维度变化 | 不变 | **+1** |
| 形状例子（两个 `(4,)`） | `(8,)` | `(2,4)` 或 `(4,2)` |
| 适用场景 | 数据本身要连起来 | 把多个样本「打包」成新的批次维度 |

> 💡 **直觉**：`concatenate` 是把两张纸**边对边**接长；`stack` 是把两张纸**叠成一摞**——多出一个「层数」维度。

In [17]:
# stack: 沿新轴堆叠 (维度 +1) vs concatenate: 沿已有轴 (维度不变)
a = np.arange(4)       # 形状 (4,)
b = np.arange(4, 8)    # 形状 (4,)

s0 = np.stack([a, b], axis=0)   # 新轴在最前: (2, 4)
s1 = np.stack([a, b], axis=1)   # 新轴在最后: (4, 2)
c0 = np.concatenate([a, b])     # 沿已有轴: (8,)
print("a.shape =", a.shape, "| b.shape =", b.shape)
print("np.stack axis=0 ->", s0.shape)
print("np.stack axis=1 ->", s1.shape)
print("np.concatenate  ->", c0.shape)
print("stack axis=0 结果:\n", s0)

a.shape = (4,) | b.shape = (4,)
np.stack axis=0 -> (2, 4)
np.stack axis=1 -> (4, 2)
np.concatenate  -> (8,)
stack axis=0 结果:
 [[0 1 2 3]
 [4 5 6 7]]


## 4.8 拆分：split / array_split / hsplit / vsplit

### 解决什么问题？

拼接的反操作：把一个数组**切成几块**。

| API | 规则 | 结果 |
| --- | --- | --- |
| `np.split(arr, n)` | **必须等分** | 等长的 n 块；不能整除会**报错** |
| `np.array_split(arr, n)` | **允许不等分** | 尽量均匀的 n 块，永不报错 |
| `np.hsplit(mat, n)` | 沿列方向切 | 二维数组竖切成 n 块 |
| `np.vsplit(mat, n)` | 沿行方向切 | 二维数组横切成 n 块 |

> 💡 **注意**：`split` 系列返回的是**视图列表**，不是拷贝——切出来的块与母数组共享内存。

In [18]:
# np.split: 必须等分
arr = np.arange(10)
parts = np.split(arr, 5)     # 10 个元素等分成 5 份
print("等分成 5 份:", [p.tolist() for p in parts])
print("split 返回视图:", np.shares_memory(arr, parts[0]))

等分成 5 份: [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]
split 返回视图: True


In [19]:
# array_split: 允许不等分; hsplit / vsplit 按轴切
arr = np.arange(10)
uneven = np.array_split(arr, 3)     # 10 个元素分 3 份 -> 4, 3, 3
print("array_split 3 份:", [p.tolist() for p in uneven])

mat = np.arange(12).reshape(3, 4)
print("hsplit(mat, 2) 竖切成 2 块:", [p.shape for p in np.hsplit(mat, 2)])
print("vsplit(mat, 3) 横切成 3 块:", [p.shape for p in np.vsplit(mat, 3)])

array_split 3 份: [[0, 1, 2, 3], [4, 5, 6], [7, 8, 9]]
hsplit(mat, 2) 竖切成 2 块: [(3, 2), (3, 2)]
vsplit(mat, 3) 横切成 3 块: [(1, 4), (1, 4), (1, 4)]


## 4.9 tile vs repeat：整体复制 vs 逐元素重复

### 为什么容易混？

两者都能把数组「变多」，但方式截然不同：

- `np.tile(a, n)`：把**整个数组**当作一块瓷砖，整体复制 n 次（贴瓷砖）
- `np.repeat(a, n)`：把**每个元素**各自重复 n 次（逐元素复印）

```mermaid
flowchart LR
    A["arr = [1, 2, 3]"] --> T["np.tile(arr, 2)<br/>整体复制(贴瓷砖)<br/>→ [1,2,3,1,2,3]"]
    A --> R["np.repeat(arr, 2)<br/>逐元素重复<br/>→ [1,1,2,2,3,3]"]
```

| | `np.tile` | `np.repeat` |
| --- | --- | --- |
| 复制单位 | 整个数组 | 单个元素 |
| `[1,2,3]` 配 `2` | `[1,2,3,1,2,3]` | `[1,1,2,2,3,3]` |
| 常见用途 | 把整块数据铺开 | 给每个样本复制副本 |

In [20]:
# tile: 整体复制; repeat: 逐元素重复
v = np.array([1, 2, 3])
print("v =", v)
print("np.tile(v, 2)         =", np.tile(v, 2))      # [1 2 3 1 2 3]
print("np.repeat(v, 2)       =", np.repeat(v, 2))    # [1 1 2 2 3 3]
print("np.repeat(v, [1,2,3]) =", np.repeat(v, [1, 2, 3]))  # 每个元素重复不同次数

v = [1 2 3]
np.tile(v, 2)         = [1 2 3 1 2 3]
np.repeat(v, 2)       = [1 1 2 2 3 3]
np.repeat(v, [1,2,3]) = [1 2 2 3 3 3]


In [21]:
# 二维下更直观
mat = np.array([[1, 2], [3, 4]])
print("mat =\n", mat)
print("tile(mat, (2,3)) 整体铺成 2 行 3 列:\n", np.tile(mat, (2, 3)))
print("repeat(mat, 2, axis=0) 每行重复:\n", np.repeat(mat, 2, axis=0))
print("repeat(mat, 2, axis=1) 每列重复:\n", np.repeat(mat, 2, axis=1))

mat =
 [[1 2]
 [3 4]]
tile(mat, (2,3)) 整体铺成 2 行 3 列:
 [[1 2 1 2 1 2]
 [3 4 3 4 3 4]
 [1 2 1 2 1 2]
 [3 4 3 4 3 4]]
repeat(mat, 2, axis=0) 每行重复:
 [[1 2]
 [1 2]
 [3 4]
 [3 4]]
repeat(mat, 2, axis=1) 每列重复:
 [[1 1 2 2]
 [3 3 4 4]]


## 4.10 实战小案例

### 案例 A：批量数据整形与还原

神经网络 / 批处理里常见这种操作：
一批数据形状 `(2, 3, 4)`（2 个批次、每批 3 个样本、每个样本 4 个特征），
先 `reshape(6, 4)` 摊平当成一个 6×4 的大表格处理，算完再 `reshape(2, 3, 4)` 拼回去。

> 💡 关键是：reshape 只是「换读法」，只要元素总数不变、顺序一致，就能无损还原。

### 案例 B：用 newaxis + concatenate 给表格加一列

一张 `(3, 2)` 的表格，想在最左边加一列「编号」（一维数组 `(3,)`）：
步骤：`ids[:, np.newaxis]` 变成 `(3, 1)` 列向量 → `concatenate([id_col, table], axis=1)`。

In [22]:
# 案例A: 把 (2,3,4) 批量数据 reshape 成 (6,4), 再拼回去
batch = np.arange(24).reshape(2, 3, 4)   # 2 批 × 3 样本 × 4 特征
print("batch.shape =", batch.shape)
flat = batch.reshape(6, 4)               # 摊平成 6 行 × 4 特征
print("reshape(6, 4).shape =", flat.shape)
back = flat.reshape(2, 3, 4)             # 拼回去
print("再 reshape 回去 =", back.shape)
print("还原成功:", np.array_equal(batch, back))

batch.shape = (2, 3, 4)
reshape(6, 4).shape = (6, 4)
再 reshape 回去 = (2, 3, 4)
还原成功: True


In [23]:
# 案例B: 用 newaxis + concatenate 给表格加一列
table = np.array([[1, 2],
                  [3, 4],
                  [5, 6]])              # 3 行 2 列
ids = np.array([100, 200, 300])         # 新的一列, (3,)
id_col = ids[:, np.newaxis]             # 变成 (3, 1) 列向量
new_table = np.concatenate([id_col, table], axis=1)   # 拼到最左边
print("id_col.shape =", id_col.shape)
print("新表格 (3,3):\n", new_table)

id_col.shape = (3, 1)
新表格 (3,3):
 [[100   1   2]
 [200   3   4]
 [300   5   6]]


## 4.11 本章小结

### 一句话记忆表

| 主题 | 一句话 |
| --- | --- |
| axis | axis 就是维度编号，axis=k 表示沿第 k 维操作 |
| reshape | 换形状不换数据；`-1` 自动推断；通常是视图 |
| 展平 | `ravel` / `reshape(-1)` 视图，`flatten` 拷贝 |
| 转置 | `.T` / `transpose` 重排轴的顺序，默认是视图 |
| 增删维度 | `newaxis` / `expand_dims` 插入长度 1 的轴，`squeeze` 删除 |
| 拼接 | `concatenate` 沿已有轴，`stack` 沿新轴（维度+1） |
| 拆分 | `split` 必须等分，`array_split` 允许不等分 |
| 复制 | `tile` 整体贴瓷砖，`repeat` 逐元素重复 |

```mermaid
mindmap
  root((第4章 形状))
    axis 心智模型
      "维度编号"
      "axis=k 沿第k维"
    reshape
      "-1 自动推断"
      "视图 不复制"
    ravel flatten
      "视图 vs 拷贝"
    transpose
      "轴重排"
      "swapaxes moveaxis"
    newaxis squeeze
      "增删长度1的轴"
      "参与广播"
    concatenate stack
      "已有轴 vs 新轴"
    split
      "等分 vs 不等分"
    tile repeat
      "整体 vs 逐元素"
```

### 📝 动手练习

1. 给定 `x = np.arange(12)`，用一次 `reshape` 把它变成 `(2, 3, 2)`，再用 `ravel` 恢复成 `(12,)`，验证恢复后与 `x` 相等。
2. 有两个形状 `(2, 4)` 的数组，分别用 `concatenate` 与 `stack` 拼起来，写出两种结果各自的形状，并解释为什么一个是 `(2, 8)` 而另一个是 `(2, 2, 4)`。
3. 用 `tile` 和 `repeat` 分别把 `np.array([1, 2])` 变成 `[1, 2, 1, 2]` 和 `[1, 1, 2, 2]`。

> 👉 **下一章：`05_统计聚合与线性代数.ipynb`**
> 形状玩得转之后，我们开始「算总账」：`sum` / `mean` / `std` 等聚合、矩阵乘法 `@`、以及轴参数 axis 在聚合中的终极用法。